In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

folder = "/content/drive/MyDrive"

for root, dirs, files in os.walk(folder):
    for file in files:
        if "rcep" in file.lower() or "trade" in file.lower() or file.endswith(".csv"):
            print(os.path.join(root, file))

/content/drive/MyDrive/通过 Chrome 保存/apartments.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2022.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2023.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2024.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/BACI_HS22_Y2022_V202601.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/BACI_HS22_Y2023_V202601.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/BACI_HS22_Y2024_V202601.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/country_codes_V202601.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/product_codes_HS22_V202601.csv
/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/01_Build_RCEP_Trade_Network.ipynb


In [ ]:
file_path = "/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2024.csv"

edges = pd.read_csv(file_path)

edges.head()

,Source,Target,Weight
0,36,96,6.215203e+05
1,36,104,1.001269e+05
2,36,116,1.148669e+05
3,36,156,1.253811e+08
4,36,360,1.007384e+07


In [ ]:
edges.head()

,Source,Target,Weight
0,36,96,6.215203e+05
1,36,104,1.001269e+05
2,36,116,1.148669e+05
3,36,156,1.253811e+08
4,36,360,1.007384e+07


In [ ]:
code_path = "/content/drive/MyDrive/RCEP_Trade_Network_Project/01_Data_Raw/country_codes_V202601.csv"

codes = pd.read_csv(code_path)

codes.head()

,country_code,country_name,country_iso2,country_iso3
0,4,Afghanistan,AF,AFG
1,8,Albania,AL,ALB
2,12,Algeria,DZ,DZA
3,16,American Samoa,AS,ASM
4,20,Andorra,AD,AND


In [ ]:
codes.columns

Index(['country_code', 'country_name', 'country_iso2', 'country_iso3'], dtype='object')

In [ ]:
code_map = codes[["country_code", "country_name"]].copy()

code_map = code_map.rename(columns={
    "country_code": "Code",
    "country_name": "Country"
})

code_map.head()

,Code,Country
0,4,Afghanistan
1,8,Albania
2,12,Algeria
3,16,American Samoa
4,20,Andorra


In [ ]:
edges_named = edges.merge(
    code_map,
    left_on="Source",
    right_on="Code",
    how="left"
)

edges_named = edges_named.rename(columns={
    "Country": "Source_Name"
})

edges_named = edges_named.drop(columns=["Code"])

In [ ]:
edges_named = edges_named.merge(
    code_map,
    left_on="Target",
    right_on="Code",
    how="left"
)

edges_named = edges_named.rename(columns={
    "Country": "Target_Name"
})

edges_named = edges_named.drop(columns=["Code"])

In [ ]:
edges_named.head()

,Source,Target,Weight,Source_Name,Target_Name
0,36,96,6.215203e+05,Australia,Brunei Darussalam
1,36,104,1.001269e+05,Australia,Myanmar
2,36,116,1.148669e+05,Australia,Cambodia
3,36,156,1.253811e+08,Australia,China
4,36,360,1.007384e+07,Australia,Indonesia


In [ ]:
edges_named[["Source_Name", "Target_Name"]].isna().sum()

,0
Source_Name,0
Target_Name,0


In [ ]:
edges_named_clean = edges_named[["Source_Name", "Target_Name", "Weight"]].copy()

edges_named_clean = edges_named_clean.rename(columns={
    "Source_Name": "Source",
    "Target_Name": "Target"
})

edges_named_clean.head()

,Source,Target,Weight
0,Australia,Brunei Darussalam,6.215203e+05
1,Australia,Myanmar,1.001269e+05
2,Australia,Cambodia,1.148669e+05
3,Australia,China,1.253811e+08
4,Australia,Indonesia,1.007384e+07


In [ ]:
output_path = "/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2024_Named.csv"

edges_named_clean.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2024_Named.csv


In [ ]:
G = nx.from_pandas_edgelist(
    edges_named_clean,
    source="Source",
    target="Target",
    edge_attr="Weight",
    create_using=nx.DiGraph()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 15
Edges: 207


In [ ]:
density = nx.density(G)

print("Network Density:", density)

Network Density: 0.9857142857142858


In [ ]:
degree = dict(G.degree())
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

weighted_in_degree = dict(G.in_degree(weight="Weight"))
weighted_out_degree = dict(G.out_degree(weight="Weight"))

degree_centrality = nx.degree_centrality(G)
in_degree_centrality = nx.in_degree_centrality(G)
out_degree_centrality = nx.out_degree_centrality(G)

betweenness = nx.betweenness_centrality(G, weight="Weight", normalized=True)
closeness = nx.closeness_centrality(G)
eigenvector = nx.eigenvector_centrality(G, weight="Weight", max_iter=1000)

In [ ]:
metrics = pd.DataFrame({
    "Country": list(G.nodes()),
    "Degree": [degree[n] for n in G.nodes()],
    "InDegree": [in_degree[n] for n in G.nodes()],
    "OutDegree": [out_degree[n] for n in G.nodes()],
    "WeightedInDegree": [weighted_in_degree[n] for n in G.nodes()],
    "WeightedOutDegree": [weighted_out_degree[n] for n in G.nodes()],
    "DegreeCentrality": [degree_centrality[n] for n in G.nodes()],
    "InDegreeCentrality": [in_degree_centrality[n] for n in G.nodes()],
    "OutDegreeCentrality": [out_degree_centrality[n] for n in G.nodes()],
    "Betweenness": [betweenness[n] for n in G.nodes()],
    "Closeness": [closeness[n] for n in G.nodes()],
    "Eigenvector": [eigenvector[n] for n in G.nodes()]
})

metrics.head()

,Country,Degree,InDegree,OutDegree,WeightedInDegree,WeightedOutDegree,DegreeCentrality,InDegreeCentrality,OutDegreeCentrality,Betweenness,Closeness,Eigenvector
0,Australia,28,14,14,1.666313e+08,2.525061e+08,2.000000,1.0,1.000000,0.000000,1.0,0.195267
1,Brunei Darussalam,27,14,13,5.295598e+06,1.037005e+07,1.928571,1.0,0.928571,0.368132,1.0,0.003960
2,Myanmar,28,14,14,1.783021e+07,1.187184e+07,2.000000,1.0,1.000000,0.412088,1.0,0.019449
3,Cambodia,28,14,14,3.366329e+07,1.084057e+07,2.000000,1.0,1.000000,0.071429,1.0,0.037361
4,China,28,14,14,7.583197e+08,8.972394e+08,2.000000,1.0,1.000000,0.000000,1.0,0.592769


In [ ]:
metrics.sort_values("WeightedOutDegree", ascending=False)

,Country,Degree,InDegree,OutDegree,WeightedInDegree,WeightedOutDegree,DegreeCentrality,InDegreeCentrality,OutDegreeCentrality,Betweenness,Closeness,Eigenvector
4,China,28,14,14,7.583197e+08,8.972394e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.592769
7,Rep. of Korea,28,14,14,2.890795e+08,3.081678e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.354914
6,Japan,28,14,14,3.602364e+08,2.885702e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.405435
0,Australia,28,14,14,1.666313e+08,2.525061e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.195267
9,Malaysia,28,14,14,1.803391e+08,2.327394e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.200264
13,Viet Nam,26,13,13,2.964563e+08,1.965867e+08,1.857143,0.928571,0.928571,0.000000,0.933333,0.378306
12,Singapore,28,14,14,2.029487e+08,1.700526e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.202358
5,Indonesia,28,14,14,1.600057e+08,1.664345e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.187053
14,Thailand,28,14,14,1.753904e+08,1.622302e+08,2.000000,1.000000,1.000000,0.000000,1.000000,0.211765
11,Philippines,28,14,14,1.027717e+08,4.569231e+07,2.000000,1.000000,1.000000,0.000000,1.000000,0.115228


In [ ]:
metrics.sort_values("WeightedOutDegree", ascending=False)[
    ["Country", "WeightedOutDegree", "OutDegree", "OutDegreeCentrality"]
]

,Country,WeightedOutDegree,OutDegree,OutDegreeCentrality
4,China,8.972394e+08,14,1.000000
7,Rep. of Korea,3.081678e+08,14,1.000000
6,Japan,2.885702e+08,14,1.000000
0,Australia,2.525061e+08,14,1.000000
9,Malaysia,2.327394e+08,14,1.000000
13,Viet Nam,1.965867e+08,13,0.928571
12,Singapore,1.700526e+08,14,1.000000
5,Indonesia,1.664345e+08,14,1.000000
14,Thailand,1.622302e+08,14,1.000000
11,Philippines,4.569231e+07,14,1.000000


In [ ]:
metrics.sort_values("WeightedInDegree", ascending=False)[
    ["Country", "WeightedInDegree", "InDegree", "InDegreeCentrality"]
]

,Country,WeightedInDegree,InDegree,InDegreeCentrality
4,China,7.583197e+08,14,1.000000
6,Japan,3.602364e+08,14,1.000000
13,Viet Nam,2.964563e+08,13,0.928571
7,Rep. of Korea,2.890795e+08,14,1.000000
12,Singapore,2.029487e+08,14,1.000000
9,Malaysia,1.803391e+08,14,1.000000
14,Thailand,1.753904e+08,14,1.000000
0,Australia,1.666313e+08,14,1.000000
5,Indonesia,1.600057e+08,14,1.000000
11,Philippines,1.027717e+08,14,1.000000


In [ ]:
metrics.sort_values("Betweenness", ascending=False)[
    ["Country", "Betweenness"]
]

,Country,Betweenness
8,Lao People's Dem. Rep.,0.730769
2,Myanmar,0.412088
1,Brunei Darussalam,0.368132
10,New Zealand,0.076923
3,Cambodia,0.071429
0,Australia,0.000000
4,China,0.000000
6,Japan,0.000000
5,Indonesia,0.000000
7,Rep. of Korea,0.000000


In [ ]:
output_path = "/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2024.csv"

metrics.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2024.csv


In [ ]:
for u, v, d in G.edges(data=True):
    d["Distance"] = 1 / d["Weight"]

betweenness_distance = nx.betweenness_centrality(
    G,
    weight="Distance",
    normalized=True
)

metrics["Betweenness_Distance"] = [
    betweenness_distance[n] for n in G.nodes()
]

metrics.sort_values("Betweenness_Distance", ascending=False)[
    ["Country", "Betweenness_Distance"]
]

,Country,Betweenness_Distance
4,China,0.961538
0,Australia,0.071429
9,Malaysia,0.071429
14,Thailand,0.071429
13,Viet Nam,0.071429
2,Myanmar,0.000000
1,Brunei Darussalam,0.000000
6,Japan,0.000000
5,Indonesia,0.000000
3,Cambodia,0.000000


In [ ]:
output_path = "/content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2024.csv"

metrics.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)

Saved to: /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2024.csv


In [ ]:
print(metrics.describe())

          Degree   InDegree  OutDegree  WeightedInDegree  WeightedOutDegree  \
count  15.000000  15.000000  15.000000      1.500000e+01       1.500000e+01   
mean   27.600000  13.800000  13.800000      1.858319e+08       1.858319e+08   
std     0.910259   0.560612   0.414039      1.944087e+08       2.248762e+08   
min    25.000000  12.000000  13.000000      5.295598e+06       8.552333e+06   
25%    28.000000  14.000000  14.000000      3.186325e+07       1.874789e+07   
50%    28.000000  14.000000  14.000000      1.666313e+08       1.664345e+08   
75%    28.000000  14.000000  14.000000      2.460141e+08       2.426228e+08   
max    28.000000  14.000000  14.000000      7.583197e+08       8.972394e+08   

       DegreeCentrality  InDegreeCentrality  OutDegreeCentrality  Betweenness  \
count         15.000000           15.000000            15.000000    15.000000   
mean           1.971429            0.985714             0.985714     0.110623   
std            0.065018            0.040044  

In [ ]:
years = [2022, 2023, 2024]

base_dir = "/content/drive/MyDrive/RCEP_Trade_Network_Project"
clean_dir = base_dir + "/02_Data_Clean"
raw_dir = base_dir + "/01_Data_Raw"

code_path = raw_dir + "/country_codes_V202601.csv"
codes = pd.read_csv(code_path)

code_map = codes[["country_code", "country_name"]].copy()
code_map = code_map.rename(columns={
    "country_code": "Code",
    "country_name": "Country"
})

all_metrics = []
network_summary = []

for year in years:
    print("Processing year:", year)

    file_path = f"{clean_dir}/RCEP_EdgeList_{year}.csv"
    edges = pd.read_csv(file_path)

    # Source code → country name
    edges_named = edges.merge(
        code_map,
        left_on="Source",
        right_on="Code",
        how="left"
    )
    edges_named = edges_named.rename(columns={"Country": "Source_Name"})
    edges_named = edges_named.drop(columns=["Code"])

    # Target code → country name
    edges_named = edges_named.merge(
        code_map,
        left_on="Target",
        right_on="Code",
        how="left"
    )
    edges_named = edges_named.rename(columns={"Country": "Target_Name"})
    edges_named = edges_named.drop(columns=["Code"])

    # keep clean edge list
    edges_named_clean = edges_named[["Source_Name", "Target_Name", "Weight"]].copy()
    edges_named_clean = edges_named_clean.rename(columns={
        "Source_Name": "Source",
        "Target_Name": "Target"
    })

    # save named edge list
    named_output_path = f"{clean_dir}/RCEP_EdgeList_{year}_Named.csv"
    edges_named_clean.to_csv(named_output_path, index=False, encoding="utf-8-sig")

    # build directed weighted network
    G = nx.from_pandas_edgelist(
        edges_named_clean,
        source="Source",
        target="Target",
        edge_attr="Weight",
        create_using=nx.DiGraph()
    )

    # network-level summary
    density = nx.density(G)

    network_summary.append({
        "Year": year,
        "Nodes": G.number_of_nodes(),
        "Edges": G.number_of_edges(),
        "Density": density
    })

    # node-level metrics
    degree = dict(G.degree())
    in_degree = dict(G.in_degree())
    out_degree = dict(G.out_degree())

    weighted_in_degree = dict(G.in_degree(weight="Weight"))
    weighted_out_degree = dict(G.out_degree(weight="Weight"))

    degree_centrality = nx.degree_centrality(G)
    in_degree_centrality = nx.in_degree_centrality(G)
    out_degree_centrality = nx.out_degree_centrality(G)

    # distance = 1 / trade value
    for u, v, d in G.edges(data=True):
        d["Distance"] = 1 / d["Weight"]

    betweenness_distance = nx.betweenness_centrality(
        G,
        weight="Distance",
        normalized=True
    )

    closeness = nx.closeness_centrality(G, distance="Distance")

    eigenvector = nx.eigenvector_centrality(
        G,
        weight="Weight",
        max_iter=1000
    )

    metrics_year = pd.DataFrame({
        "Year": year,
        "Country": list(G.nodes()),
        "Degree": [degree[n] for n in G.nodes()],
        "InDegree": [in_degree[n] for n in G.nodes()],
        "OutDegree": [out_degree[n] for n in G.nodes()],
        "WeightedInDegree": [weighted_in_degree[n] for n in G.nodes()],
        "WeightedOutDegree": [weighted_out_degree[n] for n in G.nodes()],
        "DegreeCentrality": [degree_centrality[n] for n in G.nodes()],
        "InDegreeCentrality": [in_degree_centrality[n] for n in G.nodes()],
        "OutDegreeCentrality": [out_degree_centrality[n] for n in G.nodes()],
        "Betweenness": [betweenness_distance[n] for n in G.nodes()],
        "Closeness": [closeness[n] for n in G.nodes()],
        "Eigenvector": [eigenvector[n] for n in G.nodes()]
    })

    metrics_output_path = f"{clean_dir}/RCEP_Network_Metrics_{year}.csv"
    metrics_year.to_csv(metrics_output_path, index=False, encoding="utf-8-sig")

    all_metrics.append(metrics_year)

print("Done!")

Processing year: 2022
Processing year: 2023
Processing year: 2024
Done!


In [ ]:
all_metrics_df = pd.concat(all_metrics, ignore_index=True)

all_metrics_output_path = f"{clean_dir}/RCEP_Network_Metrics_AllYears.csv"
all_metrics_df.to_csv(all_metrics_output_path, index=False, encoding="utf-8-sig")

all_metrics_df.head()

,Year,Country,Degree,InDegree,OutDegree,WeightedInDegree,WeightedOutDegree,DegreeCentrality,InDegreeCentrality,OutDegreeCentrality,Betweenness,Closeness,Eigenvector
0,2022,Australia,28,14,14,1.796118e+08,3.133420e+08,2.000000,1.000000,1.000000,0.071429,9.156338e+06,0.201820
1,2022,Brunei Darussalam,26,13,13,3.955030e+06,1.342423e+07,1.857143,0.928571,0.928571,0.000000,1.502766e+06,0.003066
2,2022,Myanmar,28,14,14,2.517775e+07,1.606838e+07,2.000000,1.000000,1.000000,0.000000,5.687662e+06,0.026752
3,2022,Cambodia,28,14,14,4.029370e+07,8.270835e+06,2.000000,1.000000,1.000000,0.000000,6.690656e+06,0.038138
4,2022,China,28,14,14,7.640423e+08,9.211085e+08,2.000000,1.000000,1.000000,0.939560,1.009921e+07,0.585565


In [ ]:
network_summary_df = pd.DataFrame(network_summary)

summary_output_path = f"{clean_dir}/RCEP_Network_Summary_2022_2024.csv"
network_summary_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

network_summary_df

,Year,Nodes,Edges,Density
0,2022,15,208,0.990476
1,2023,15,210,1.000000
2,2024,15,207,0.985714


In [ ]:
all_metrics_df[
    all_metrics_df["Country"].isin(["China", "Japan", "Rep. of Korea"])
].sort_values(["Country", "Year"])[
    ["Year", "Country", "WeightedOutDegree", "WeightedInDegree", "Betweenness", "Eigenvector"]
]

,Year,Country,WeightedOutDegree,WeightedInDegree,Betweenness,Eigenvector
4,2022,China,9.211085e+08,7.640423e+08,0.939560,0.585565
19,2023,China,8.514387e+08,7.068171e+08,0.956044,0.584789
34,2024,China,8.972394e+08,7.583197e+08,0.961538,0.592769
6,2022,Japan,3.368543e+08,4.346319e+08,0.000000,0.446308
21,2023,Japan,2.993369e+08,3.806865e+08,0.000000,0.439165
36,2024,Japan,2.885702e+08,3.602364e+08,0.000000,0.405435
7,2022,Rep. of Korea,3.425174e+08,3.212879e+08,0.000000,0.368103
22,2023,Rep. of Korea,2.870843e+08,2.928797e+08,0.000000,0.374094
37,2024,Rep. of Korea,3.081678e+08,2.890795e+08,0.000000,0.354914


In [ ]:
print("=" * 50)
print("✅ Network analysis completed successfully!")
print("=" * 50)

print("\nGenerated files:")
print(f"- {clean_dir}/RCEP_EdgeList_2022_Named.csv")
print(f"- {clean_dir}/RCEP_EdgeList_2023_Named.csv")
print(f"- {clean_dir}/RCEP_EdgeList_2024_Named.csv")

print(f"- {clean_dir}/RCEP_Network_Metrics_2022.csv")
print(f"- {clean_dir}/RCEP_Network_Metrics_2023.csv")
print(f"- {clean_dir}/RCEP_Network_Metrics_2024.csv")

print(f"- {clean_dir}/RCEP_Network_Metrics_AllYears.csv")
print(f"- {clean_dir}/RCEP_Network_Summary_2022_2024.csv")

print("\nSummary:")
print(network_summary_df)

✅ Network analysis completed successfully!

Generated files:
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2022_Named.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2023_Named.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_EdgeList_2024_Named.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2022.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2023.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_2024.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Metrics_AllYears.csv
- /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Network_Summary_2022_2024.csv

Summary:
   Year  Nodes  Edges   Density
0  2022     15    208  0.990476
1  2023     15    210  1.000000
2  2024     15    207  0.985714


In [ ]:
table1 = network_summary_df.copy()

table1 = table1.rename(columns={
    "Year":"Year",
    "Nodes":"Number of Nodes",
    "Edges":"Number of Edges",
    "Density":"Network Density"
})

table1

,Year,Number of Nodes,Number of Edges,Network Density
0,2022,15,208,0.990476
1,2023,15,210,1.000000
2,2024,15,207,0.985714


In [ ]:
table2 = all_metrics_df[
    all_metrics_df["Country"].isin(
        ["China","Japan","Rep. of Korea"]
    )
][[
    "Year",
    "Country",
    "WeightedOutDegree",
    "WeightedInDegree",
    "Betweenness",
    "Eigenvector"
]]

table2

,Year,Country,WeightedOutDegree,WeightedInDegree,Betweenness,Eigenvector
4,2022,China,9.211085e+08,7.640423e+08,0.939560,0.585565
6,2022,Japan,3.368543e+08,4.346319e+08,0.000000,0.446308
7,2022,Rep. of Korea,3.425174e+08,3.212879e+08,0.000000,0.368103
19,2023,China,8.514387e+08,7.068171e+08,0.956044,0.584789
21,2023,Japan,2.993369e+08,3.806865e+08,0.000000,0.439165
22,2023,Rep. of Korea,2.870843e+08,2.928797e+08,0.000000,0.374094
34,2024,China,8.972394e+08,7.583197e+08,0.961538,0.592769
36,2024,Japan,2.885702e+08,3.602364e+08,0.000000,0.405435
37,2024,Rep. of Korea,3.081678e+08,2.890795e+08,0.000000,0.354914


In [ ]:
top5_export = (
    all_metrics_df
    .sort_values(
        ["Year","WeightedOutDegree"],
        ascending=[True,False]
    )
    .groupby("Year")
    .head(5)
)

top5_export[
    ["Year","Country","WeightedOutDegree"]
]

,Year,Country,WeightedOutDegree
4,2022,China,9.211085e+08
7,2022,Rep. of Korea,3.425174e+08
6,2022,Japan,3.368543e+08
0,2022,Australia,3.133420e+08
9,2022,Malaysia,2.231081e+08
19,2023,China,8.514387e+08
21,2023,Japan,2.993369e+08
22,2023,Rep. of Korea,2.870843e+08
15,2023,Australia,2.840609e+08
24,2023,Malaysia,1.987764e+08


In [ ]:
top5_import = (
    all_metrics_df
    .sort_values(
        ["Year","WeightedInDegree"],
        ascending=[True,False]
    )
    .groupby("Year")
    .head(5)
)

top5_import[
    ["Year","Country","WeightedInDegree"]
]

,Year,Country,WeightedInDegree
4,2022,China,7.640423e+08
6,2022,Japan,4.346319e+08
7,2022,Rep. of Korea,3.212879e+08
13,2022,Viet Nam,2.748922e+08
12,2022,Singapore,2.243801e+08
19,2023,China,7.068171e+08
21,2023,Japan,3.806865e+08
22,2023,Rep. of Korea,2.928797e+08
28,2023,Viet Nam,2.439694e+08
27,2023,Singapore,2.027289e+08


In [ ]:
top5_between = (
    all_metrics_df
    .sort_values(
        ["Year","Betweenness"],
        ascending=[True,False]
    )
    .groupby("Year")
    .head(5)
)

top5_between[
    ["Year","Country","Betweenness"]
]

,Year,Country,Betweenness
4,2022,China,0.939560
14,2022,Thailand,0.142857
0,2022,Australia,0.071429
9,2022,Malaysia,0.071429
13,2022,Viet Nam,0.071429
19,2023,China,0.956044
15,2023,Australia,0.071429
24,2023,Malaysia,0.071429
28,2023,Viet Nam,0.071429
29,2023,Thailand,0.071429


In [ ]:
# Table 1: Network Summary
table1 = network_summary_df.copy()

# Table 2: China, Japan, Korea
table2 = all_metrics_df[
    all_metrics_df["Country"].isin(["China", "Japan", "Rep. of Korea"])
][[
    "Year", "Country", "WeightedOutDegree", "WeightedInDegree", "Betweenness", "Eigenvector"
]].sort_values(["Country", "Year"])

# Table 3: Top 5 Export
table3 = (
    all_metrics_df
    .sort_values(["Year", "WeightedOutDegree"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "WeightedOutDegree"]]

# Table 4: Top 5 Import
table4 = (
    all_metrics_df
    .sort_values(["Year", "WeightedInDegree"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "WeightedInDegree"]]

# Table 5: Top 5 Betweenness
table5 = (
    all_metrics_df
    .sort_values(["Year", "Betweenness"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "Betweenness"]]

In [ ]:
excel_path = f"{clean_dir}/RCEP_Report_Tables.xlsx"

with pd.ExcelWriter(excel_path) as writer:
    table1.to_excel(writer, sheet_name="Network Summary", index=False)
    table2.to_excel(writer, sheet_name="China Japan Korea", index=False)
    table3.to_excel(writer, sheet_name="Top5 Export", index=False)
    table4.to_excel(writer, sheet_name="Top5 Import", index=False)
    table5.to_excel(writer, sheet_name="Top5 Betweenness", index=False)

print("Saved to:", excel_path)

Saved to: /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Report_Tables.xlsx


In [ ]:
# ===== Generate Final Excel for Report =====

excel_path = f"{clean_dir}/RCEP_Report_Tables_Final.xlsx"

# 1. Network Density / overall summary
table_network_summary = network_summary_df.copy()

# 2. All node-level metrics: 2022–2024 dynamic comparison
table_all_metrics = all_metrics_df[[
    "Year",
    "Country",
    "Degree",
    "InDegree",
    "OutDegree",
    "WeightedInDegree",
    "WeightedOutDegree",
    "DegreeCentrality",
    "InDegreeCentrality",
    "OutDegreeCentrality",
    "Betweenness",
    "Closeness",
    "Eigenvector"
]].copy()

# 3. China / Japan / Korea comparison
table_china_japan_korea = table_all_metrics[
    table_all_metrics["Country"].isin(["China", "Japan", "Rep. of Korea"])
].sort_values(["Country", "Year"])

# 4. Top 5 Weighted Export
table_top5_export = (
    table_all_metrics
    .sort_values(["Year", "WeightedOutDegree"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "WeightedOutDegree"]]

# 5. Top 5 Weighted Import
table_top5_import = (
    table_all_metrics
    .sort_values(["Year", "WeightedInDegree"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "WeightedInDegree"]]

# 6. Top 5 Betweenness
table_top5_betweenness = (
    table_all_metrics
    .sort_values(["Year", "Betweenness"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "Betweenness"]]

# 7. Top 5 Eigenvector
table_top5_eigenvector = (
    table_all_metrics
    .sort_values(["Year", "Eigenvector"], ascending=[True, False])
    .groupby("Year")
    .head(5)
)[["Year", "Country", "Eigenvector"]]

# Save everything into one Excel file
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    table_network_summary.to_excel(writer, sheet_name="Network Density", index=False)
    table_all_metrics.to_excel(writer, sheet_name="All Metrics 2022-2024", index=False)
    table_china_japan_korea.to_excel(writer, sheet_name="China Japan Korea", index=False)
    table_top5_export.to_excel(writer, sheet_name="Top5 Export", index=False)
    table_top5_import.to_excel(writer, sheet_name="Top5 Import", index=False)
    table_top5_betweenness.to_excel(writer, sheet_name="Top5 Betweenness", index=False)
    table_top5_eigenvector.to_excel(writer, sheet_name="Top5 Eigenvector", index=False)

print("Saved to:", excel_path)

Saved to: /content/drive/MyDrive/RCEP_Trade_Network_Project/02_Data_Clean/RCEP_Report_Tables_Final.xlsx


In [44]:
import pandas as pd

# 只保留贸易额前30%的边
threshold = edges_named_clean["Weight"].quantile(0.70)

edges_gephi = edges_named_clean[
    edges_named_clean["Weight"] >= threshold
].copy()

print("Original edges:", len(edges_named_clean))
print("Filtered edges:", len(edges_gephi))

Original edges: 207
Filtered edges: 62


In [45]:
edges_gephi.to_csv(
    f"{clean_dir}/RCEP_EdgeList_2024_Gephi.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!


In [47]:
gephi_dir = "/content/drive/MyDrive/RCEP_Trade_Network_Project/03_Gephi"

In [48]:
gephi_dir = "/content/drive/MyDrive/RCEP_Trade_Network_Project/03_Gephi"

G2 = nx.from_pandas_edgelist(
    edges_gephi,
    source="Source",
    target="Target",
    edge_attr="Weight",
    create_using=nx.DiGraph()
)

nx.write_gexf(
    G2,
    f"{gephi_dir}/RCEP_Gephi_2024_Filtered.gexf"
)

print("Filtered Gephi file saved!")

Filtered Gephi file saved!
